# **Step 1: Creating the Medallion Folders**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
base = "/content/drive/MyDrive/Moonstone_Instacart"
for folder in ["bronze", "silver", "gold"]:
    os.makedirs(f"{base}/{folder}", exist_ok=True)
print("✅ Bronze / Silver / Gold folders created")

Mounted at /content/drive
✅ Bronze / Silver / Gold folders created


# **Step 2: Automated Pipeline**

In [3]:
# MOONSTONE PHASE 4 PIPELINE
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, regexp_replace, lower, split, explode, size, length, count, avg
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.appName("Moonstone_Phase4_Pipeline").getOrCreate()
base = "/content/drive/MyDrive/Moonstone_Instacart/"

print("🚀 Starting Medallion Pipeline with your exact 5 files...")

🚀 Starting Medallion Pipeline with your exact 5 files...


In [4]:
# 1. BRONZE – Read files
departments = spark.read.csv(f"{base}bronze/departments.csv", header=True, inferSchema=True)
aisles      = spark.read.csv(f"{base}bronze/aisles.csv", header=True, inferSchema=True)
products    = spark.read.csv(f"{base}bronze/products.csv", header=True, inferSchema=True)
orders      = spark.read.csv(f"{base}bronze/orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv(f"{base}bronze/order_items.csv", header=True, inferSchema=True)



In [5]:
# 2. SILVER – Cleansed layer
def add_governance(df, source):
    return df.withColumn("ingestion_timestamp", current_timestamp()) \
             .withColumn("pipeline_version", lit("v1.0")) \
             .withColumn("source_layer", lit(source))

In [6]:
# Departments
silver_dept = (departments
    .withColumn("department_id", regexp_replace(col("department_id"), "[^0-9]", "").cast(IntegerType()))
    .withColumn("department_clean", lower(col("department")))
    .withColumn("tags", split(col("department_clean"), " "))
    .withColumn("tag", explode(col("tags")))
    .withColumn("dept_size_band", when(size(col("tags")) <= 1, "single-word")
                                 .when(size(col("tags")) == 2, "two-word")
                                 .otherwise("multi-word"))
)
silver_dept = add_governance(silver_dept, "bronze")
silver_dept.write.mode("overwrite").parquet(f"{base}silver/departments_clean")

# Aisles
silver_aisle = (aisles
    .withColumn("aisle_id", regexp_replace(col("aisle_id"), "[^0-9]", "").cast(IntegerType()))
    .withColumn("aisle_clean", lower(col("aisle")))
    .withColumn("tags", split(col("aisle_clean"), " "))
    .withColumn("tag", explode(col("tags")))
    .withColumn("category_size_band", when(size(col("tags")) <= 1, "single-word")
                                    .when(size(col("tags")) == 2, "two-word")
                                    .otherwise("multi-word"))
)
silver_aisle = add_governance(silver_aisle, "bronze")
silver_aisle.write.mode("overwrite").parquet(f"{base}silver/aisles_clean")

In [8]:
from pyspark.sql.types import StructType, StructField # Add these imports

# Define the schema for order_items.csv
# Assuming typical Instacart order_products.csv structure
order_items_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("add_to_cart_order", IntegerType(), True),
    StructField("reordered", IntegerType(), True)
])

# Reload order_items with the correct schema
# This assumes 'base' and 'spark' are available from previous cells.
# Note: This will overwrite the 'order_items' DataFrame loaded in an earlier cell.
order_items = spark.read.csv(f"{base}bronze/order_items.csv", header=True, schema=order_items_schema)

# Products, Orders, Order_Items (simple clean)
silver_products = products.withColumn("product_name_clean", lower(regexp_replace(col("product_name"), r'[^a-z0-9\s]', '')))
silver_products = add_governance(silver_products, "bronze")
silver_products.write.mode("overwrite").parquet(f"{base}silver/products_clean")

silver_orders = orders.withColumn("days_since_prior_order", when(col("days_since_prior_order").isNull(), 0.0).otherwise(col("days_since_prior_order").cast("float")))
silver_orders = add_governance(silver_orders, "bronze")
silver_orders.write.mode("overwrite").parquet(f"{base}silver/orders_clean")

silver_order_items = order_items.withColumn("add_to_cart_order", col("add_to_cart_order").cast(IntegerType())) \
                               .withColumn("reordered", col("reordered").cast(IntegerType()))
silver_order_items = add_governance(silver_order_items, "bronze")
silver_order_items.write.mode("overwrite").parquet(f"{base}silver/order_items_clean")

print("✅ Silver layer written")

✅ Silver layer written


In [9]:
# 3. GOLD – Curated and Business KPI Table
gold_kpi = (silver_dept
    .groupBy("department_clean", "dept_size_band")
    .agg(count("tag").alias("tag_count"),
         avg(length("tag")).alias("avg_tag_length"))
    .orderBy("tag_count", ascending=False)
)
gold_kpi.write.mode("overwrite").parquet(f"{base}gold/daily_department_kpi")


In [10]:
# Save main tables to Gold
silver_dept.write.mode("overwrite").parquet(f"{base}gold/departments_gold")
silver_aisle.write.mode("overwrite").parquet(f"{base}gold/aisles_gold")
silver_products.write.mode("overwrite").parquet(f"{base}gold/products_gold")
silver_orders.write.mode("overwrite").parquet(f"{base}gold/orders_gold")
silver_order_items.write.mode("overwrite").parquet(f"{base}gold/order_items_gold")

print("✅ GOLD layer complete!")
gold_kpi.show(5)

✅ GOLD layer complete!
+----------------+--------------+---------+-----------------+
|department_clean|dept_size_band|tag_count|   avg_tag_length|
+----------------+--------------+---------+-----------------+
| dry goods pasta|    multi-word|        3|4.333333333333333|
|    canned goods|      two-word|        2|              5.5|
|      dairy eggs|      two-word|        2|              4.5|
|    meat seafood|      two-word|        2|              5.5|
|   personal care|      two-word|        2|              6.0|
+----------------+--------------+---------+-----------------+
only showing top 5 rows
